In [1]:
import datetime as dt
import time

import pandas as pd

from utilities.preprocessor import *
from utilities.values import *

In [2]:
data_dir = './data/'
dataset_preprocessor = DatasetPreprocessor()

### 1. Merge the separate raw datafiles

In [ ]:
# combine all the raw files
raw_data_dir_paths = [data_dir + 'raw/2020', data_dir + 'raw/2021',]
full_dataset_df = dataset_preprocessor.read_all_files_in_dir(raw_data_dir_paths)

# sort by increasing start and end dates
full_dataset_df = full_dataset_df.sort_values(['starttime', 'stoptime'])
print('Total number of rows:', len(full_dataset_df))

### 2. EDA

In [ ]:
# check for missing values - None
def print_missing_values(df):
    num_rows_with_missing_values = df.isnull().any(axis=1).sum()
    print(df.isna().sum())
    print('Total rows:', len(df))
    print('Rows with missing values:', num_rows_with_missing_values)
    print('Proportion: {} %'.format(round(num_rows_with_missing_values/len(df) *100, 4)))

print_missing_values(full_dataset_df)

In [ ]:
# look at distributions

### 3. Preprocessing the dataset

#### 3a. Extract date features and get the rental counts for each station, date and time

In [ ]:
# force starttime into datetime format
full_dataset_df['starttime'] = pd.to_datetime(full_dataset_df['starttime'])

# create new date features
full_dataset_df[DATE] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_date)
full_dataset_df[YEAR] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_year)
full_dataset_df[MONTH] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_month)
full_dataset_df[WEEK_OF_YEAR] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_week_of_year)
full_dataset_df[DAY_OF_WEEK] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_day_of_week)
full_dataset_df[HOUR_OF_DAY] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_hour_of_day)
full_dataset_df[IS_PUBLIC_HOLIDAY] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_is_public_holiday)

In [ ]:
# drop columns that will not be used
columns_to_drop = [
    'tripduration', 'starttime', 'stoptime', 'start station id', 'end station id', 'end station name', 
    'end station latitude', 'end station longitude', 'bikeid', 'usertype', 'birth year', 'gender'
]
full_dataset_df = full_dataset_df.drop(columns=columns_to_drop)

In [ ]:
# sum rows for each station, date and time
full_rental_counts_df = (
    full_dataset_df
    .groupby(list(full_dataset_df.columns), dropna = False)
    .size()
    .reset_index(name = NUM_RENTALS)
)

# sort for easy reading
full_rental_counts_df = full_rental_counts_df.sort_values([STATION, DATE, HOUR_OF_DAY])

#### 3b. Create new features using the actual rental counts of the last 3 hours

In [ ]:
# timestamp of each row, so that T-1 works across midnight
rental_timestamp = (
    pd.to_datetime(full_rental_counts_df[DATE])
    + pd.to_timedelta(full_rental_counts_df[HOUR_OF_DAY], unit = 'h')
)

# lookup table: rentals for each station at each hour
rentals_lookup = (
    full_rental_counts_df
    .assign(timestamp = rental_timestamp)
    .groupby([STATION, 'timestamp'])[NUM_RENTALS]
    .sum()
)
data_start = rental_timestamp.min()

for lag in [1, 2, 3]:
    lag_timestamp = rental_timestamp - pd.Timedelta(hours = lag)
    lag_keys = pd.MultiIndex.from_arrays([full_rental_counts_df[STATION], lag_timestamp])

    # same station, exactly `lag` hours earlier; no row for that hour means 0 rentals
    lag_values = pd.Series(
        rentals_lookup.reindex(lag_keys).fillna(0).to_numpy(),
        index = full_rental_counts_df.index,
    )

    # hours before the dataset begins are unknown, not 0
    full_rental_counts_df[f'{NUM_RENTALS}_t-{lag}'] = lag_values.where(lag_timestamp >= data_start)

In [ ]:
# save to file
full_rental_counts_df.to_csv(data_dir + 'intermediate_processing/full_rental_counts.csv', index=False)
print('Total number of rows:', len(full_rental_counts_df))

#### 3c. Extract weather features

In [ ]:
# get the location and dates to query
weather_search_df = full_rental_counts_df[[STATION, LATITUDE, LONGITUDE, DATE]]
weather_search_df = weather_search_df.drop_duplicates()
stations = set(weather_search_df[STATION])

In [ ]:
# call the weather API for each location
weather_df = pd.DataFrame()

for i, station in enumerate(stations, start = 1):
    location_df = weather_search_df[weather_search_df[STATION] == station]
    location_df = location_df.sort_values(DATE)

    params = {
        START_DATE: location_df[DATE].iloc[0],
        END_DATE: location_df[DATE].iloc[-1],
        STATION: station,
        LATITUDE: location_df[LATITUDE].iloc[0],
        LONGITUDE: location_df[LONGITUDE].iloc[0]
    }

    # get historical weather data
    station_weather_history_df = dataset_preprocessor.get_hourly_weather(params)
    weather_df = pd.concat([weather_df, station_weather_history_df])

    # pause for 1 min after every 20 calls to stay under the API rate limit
    if i % 20 == 0:
        print(f"{i} stations done, sleeping for 60s...")
        time.sleep(60)

# save results
weather_df.to_csv(data_dir + 'intermediate_processing/historical_weather_features.csv', index=False)

#### 3d. Merge the citibike and weather features

In [3]:
# read previously stored intermediate files
full_rental_counts_df = pd.read_csv(data_dir + 'intermediate_processing/full_rental_counts.csv')
weather_df = pd.read_csv(data_dir + 'intermediate_processing/historical_weather_features.csv')

In [4]:
merge_columns = [STATION, DATE, HOUR_OF_DAY]

# one weather row per station-date-hour
# drops the repeated hour on 1 Nov daylight savings end
weather_to_merge_df = (
    weather_df
    .drop(columns = [LATITUDE, LONGITUDE])
    .drop_duplicates(subset = merge_columns, keep = "first")
)

In [5]:
# lag the weather by 1 hour: the weather recorded at hour T-1 is used to predict hour T
# done on a timestamp so that 11pm rolls over to 12am of the next date
lagged_timestamp = (
    pd.to_datetime(weather_to_merge_df[DATE])
    + pd.to_timedelta(weather_to_merge_df[HOUR_OF_DAY] + 1, unit = 'h')
)
weather_to_merge_df[DATE] = lagged_timestamp.dt.strftime('%Y-%m-%d')
weather_to_merge_df[HOUR_OF_DAY] = lagged_timestamp.dt.hour

In [6]:
# merge features
full_features_df = full_rental_counts_df.merge(
    weather_to_merge_df,
    on = merge_columns,
    how = "left",
    validate = "many_to_one",
)

# drop columns that will not be used
columns_to_drop = [DATE, LATITUDE, LONGITUDE]
full_features_df = full_features_df.drop(columns=columns_to_drop)


In [7]:
full_features_df.to_csv(data_dir + 'intermediate_processing/full_features.csv', index=False)

#### 3f. Perform one-hot encoding and feature normalisation

#### 3g. Train - Validation - Test split